# S9.1 · 合规证据链核验

M9 的验收人不是技术方，而是法务与合规。所以本模块的第一件事不是写文档，而是**让「每条结论追得到证据」成为可执行的检查**——写成「已核验」四个字，谁也不知道核了什么。

> ⚠️ 核验的是**可追溯性**，不是**正确性**。证据文件存在不代表结论正确，只代表读者能查到它依据的是什么。

In [1]:
ROUND_DP = 4          # 表格展示精度（不影响任何计算结果）
import sys, subprocess, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "registry").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, yaml
CONFIG_PATH = ROOT / "modules/m9_documentation/configs/evidence_map.yaml"
config = yaml.safe_load(open(CONFIG_PATH, encoding="utf-8"))
seed = config.get("seeds", [config.get("seed")])[0]
git = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                     capture_output=True, text=True, cwd=ROOT).stdout.strip()
print("config:", CONFIG_PATH.relative_to(ROOT))
print("seed  :", seed, "| 全部种子:", config.get("seeds"))
print("git   :", git or "(未提交)")
print("numpy :", np.__version__, "| pandas:", pd.__version__)

config: modules/m9_documentation/configs/evidence_map.yaml
seed  : None | 全部种子: None
git   : 3395cea
numpy : 2.3.5 | pandas: 2.3.3


In [2]:
from modules.m9_documentation.components import evidence_chain as EC
claims = EC.verify_claims(config['claims'])
risks = EC.verify_risk_traceability(config['risks'])
deliv = EC.verify_deliverables(config['deliverables'])
PCT = 100
for label, res, ok, tot in [('一致性核验', claims, 'traceable', 'total'),
                            ('DPIA 风险溯源', risks, 'traced', 'total'),
                            ('交付清单', deliv, 'present', 'total')]:
    print(f'{label:14s} {res[ok]}/{res[tot]} = {res["rate"]*PCT:.1f}%')

一致性核验          0/13 = 0.0%
DPIA 风险溯源      0/7 = 0.0%
交付清单           0/10 = 0.0%


## 逐条结论的可追溯性

`nature` 一列必须显式声明性质——**留空会让读者误以为是实测**。凡涉法结论一律标注「未复核」：本项目不设合规角色。

In [3]:
pd.DataFrame(claims['rows'])[['id', 'statement', 'nature',
                              'n_evidence', 'traceable']]

,id,statement,nature,n_evidence,traceable
0,C1,L1（本地模型 + 对方 k-匿名聚合统计）只捕获纵向联邦价值的一小部分,合成数据实测（三个独立口径：2.1% / 15.5% / 12.3%）,4,False
1,C2,价值高度依赖互补性这一个条件，λ 从 0 到 2.0 时增益跨度八倍,合成数据实测,1,False
2,C3,逐轮加高斯噪声防不住标签推断；压住泄露所需的噪声会先把价值清零,合成数据实测（含跨轮平均攻击的证伪）,2,False
3,C4,无防护的纵向联邦逻辑回归是双向完全泄露的——主动方拿特征，被动方拿标签,合成数据实测（6 个辅助样本即可精确恢复特征，R²=1.000000）,2,False
4,C5,防护必须逐攻击面设计——同一种加噪手段对标签推断与特征推断效果完全相反,合成数据实测（含加强版攻击的证伪）,3,False
5,C6,成员泄露由模型容量驱动，与是否联邦无关,合成数据实测（LiRA 校准，32 影子模型）,2,False
6,C7,合成数据上的模型效果排序是生成假设的产物，不得用于选技术路线,合成数据实测（分支卡 falsifier 触发）,2,False
7,C8,C1 不依赖数据生成机制，三种信号形式下 L1 捕获比例均 ≤1%,合成数据实测,1,False
8,C9,理论上限高不等于可实现比例高，用一个数字概括数据价值不可靠,合成数据实测,2,False
9,A1,实体对齐质量不是本项目的主要风险——漏配 10% 仅损失约 12% 增益,合成数据实测（PSI 为行为仿真，非密码学实现）,2,False


## DPIA 风险溯源

每条风险必须指向具体的实验产出，并给出缓解措施——只写「可能存在」不算溯源。

In [4]:
pd.DataFrame(risks['rows'])[['id', 'risk', 'severity',
                             'has_mitigation', 'traced']]

,id,risk,severity,has_mitigation,traced
0,R1,被动方原始特征被主动方精确恢复,高,True,False
1,R2,主动方标签被被动方完全推断,高,True,False
2,R3,成员关系泄露（某人是否在建模样本内本身即个人信息）,中,True,False
3,R4,SplitNN 形态B 的特征反演风险高于形态A,中,True,False
4,R5,实体错配导致个人信息被错误关联,中,True,False
5,R6,静默降级——被动方下线后下游把单方模型名单当作联邦模型名单,中,True,False
6,R7,schema 变更后照常出分，名单全错而无人察觉,高,True,False


**R2 是本评估最重要的一条：主动方标签被完全推断，目前无有效技术缓解。**

噪声防护已被实测证伪——把泄露压到 0.67 需要 σ=30，而那时模型可用性（0.6352）已低于不做联邦的内地单方基线（0.7089）。**防护到有效时，还不如不做联邦。**

## 交付清单

In [5]:
pd.DataFrame(deliv['rows'])[['name', 'path', 'status']]

,name,path,status
0,DPIA（数据保护影响评估）,modules/m9_documentation/DPIA.md,missing
1,对外表述红线,modules/m9_documentation/对外表述红线.md,missing
2,模型卡,modules/m9_documentation/model_card.md,missing
3,跨境流动清单,modules/m9_documentation/cross_border_flow_reg...,missing
4,审计日志规格,modules/m9_documentation/audit_log_spec.md,missing
5,同意管理规格,modules/m9_documentation/consent_management_sp...,missing
6,数据卡（合成数据）,modules/m9_documentation/data_cards/synthetic_...,missing
7,剩余风险签署,modules/m9_documentation/residual_risk_signoff...,missing
8,路线取舍决议,modules/m8_industrialization/路线取舍决议.md,missing
9,上线准入清单,modules/m8_industrialization/上线准入清单.md,missing


## 这套核验防的是什么

一类很安静的事故：有人删掉或重命名了一个结果文件，文档里引用它的那条结论就此失去依据，而**七道门禁照样全绿**。

本核验已接入 `ci/check_evidence_chain.py`，随 `run_all_gates.sh` 每次提交前执行。写这份 notebook 的过程中它就抓到过一次：证据映射里引用了一个不存在的 `funnel_scenarios.csv`，通过率因此掉到 92.3%。